# Landmark Semantic Search with ChromaDB Persistent Client

In [1]:
import chromadb
from chromadb.utils import embedding_functions

In [2]:
CHROMA_DATA_PATH = "chroma_data/"
EMBED_MODEL = "all-MiniLM-L6-v2"
COLLECTION_NAME = "FamousLandMarks"

In [3]:
client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)

embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL
)

D:\AI_Demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 34140.45it/s]

In [4]:
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_func,
    metadata={"hnsw:space": "cosine"},
)

In [5]:
documents = [
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. Built in 1889 for the World's Fair, it has become a global cultural icon of France.",
    "The Great Wall of China is a series of fortifications built across the historical northern borders of ancient Chinese states to protect against invasions. It stretches thousands of kilometres.",
    "Machu Picchu is a 15th-century Incan citadel set high in the Andes Mountains of Peru, above the Urubamba River valley. It is famed for its dry-stone walls and panoramic views.",
]
ids = ["id1", "id2", "id3"]
metadatas = [
    {"category": "tower", "country": "France", "year": 1889},
    {"category": "fortification", "country": "China", "year": -700},
    {"category": "citadel", "country": "Peru", "year": 1450},
]

# upsert keeps the cell safe to re-run against the persistent store
collection.upsert(documents=documents, ids=ids, metadatas=metadatas)

In [6]:
query_results = collection.query(
    query_texts=["Tell me about the Incan citadel."],
    n_results=1,
)

In [7]:
print("Document:", query_results["documents"][0][0])
print("Distance:", query_results["distances"][0][0])
print("Metadata:", query_results["metadatas"][0][0])
print(query_results)

Document: Machu Picchu is a 15th-century Incan citadel set high in the Andes Mountains of Peru, above the Urubamba River valley. It is famed for its dry-stone walls and panoramic views.
Distance: 0.4391457438468933
Metadata: {'category': 'citadel', 'year': 1450, 'country': 'Peru'}
{'ids': [['id3']], 'embeddings': None, 'documents': [['Machu Picchu is a 15th-century Incan citadel set high in the Andes Mountains of Peru, above the Urubamba River valley. It is famed for its dry-stone walls and panoramic views.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'citadel', 'year': 1450, 'country': 'Peru'}]], 'distances': [[0.4391457438468933]]}
